# Step 12v5: Cell Marker Visualization

This notebook extracts and visualizes individual cells from their original images, displaying marker expression patterns for phenotype validation.

## Overview

Generates visual inspection of individual cells by:

- **Sample random cells** from selected phenotypes across multiple slides
- **Extract cell boundaries** using segmentation polygons from GeoJSON files
- **Load marker intensities** from original ROI TIFF images
- **Display marker channels** individually and overlaid for visual validation
- **Generate publication-quality figures** showing marker expression patterns

## Workflow

1. Load unified CSV with cell phenotypes, coordinates, and batch IDs
2. Filter to phenotype(s) of interest
3. Randomly sample N cells from selected phenotype
4. For each cell:
   - Get x_centroid, y_centroid from CSV
   - Get batch_id to identify source slide
   - Load corresponding GeoJSON to extract cell polygon
   - Load ROI TIFF image channels
   - Extract pixels within polygon for each marker
5. Display extracted cell images with marker channels

## Input

- **Unified CSV**: `outputs/phenotyping/all_samples_combined_corrected_phenotyped.csv` (from Step 11)
- **GeoJSON boundaries**: `outputs/segmentation/cellpose/boundaries/` (from Step 3A)
- **ROI images**: `outputs/registration/roi/` (from Step 2)

## Output

- **Cell images**: PNG figures showing marker expression for individual cells
- **Summary statistics**: Number and distribution of sampled cells
- **Validation data**: Quality metrics for marker extraction

## Imports and Setup

In [ ]:
# Imports & Environment Setup
import os
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import binary_dilation
import json
import tifffile
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union
from tqdm import tqdm
import random

warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.utils import log_time

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(f"Project root: {project_root}")
print("Step 12v5: Cell Marker Visualization Setup Complete")

## Setup, Data Loading & Configuration

Load data and directory configuration. Then adjust cell extraction parameters based on available phenotypes and markers.

In [ ]:
# =============================================================================
# SETUP & CONFIGURATION: Paths and Directories
# =============================================================================

# I/O Paths
INPUT_DIR = project_root / 'outputs' / 'phenotyping'
SEGMENTATION_DIR = project_root / 'outputs' / 'segmentation' / 'cellpose' 
ROI_IMAGES_DIR = project_root / 'outputs' / 'registration' / 'roi'
OUTPUT_DIR = project_root / 'outputs' / 'spatial_figures' / 'cell_markers'

# Input file (unified phenotyping output from Step 11)
UNIFIED_INPUT_FILE = INPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded:")
print(f"  Input file: {UNIFIED_INPUT_FILE}")
print(f"  Segmentation dir: {SEGMENTATION_DIR}")
print(f"  ROI images dir: {ROI_IMAGES_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

# =============================================================================
# STEP 1: DATA LOADING
# =============================================================================
print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# Load unified CSV
print(f"\nLoading unified CSV: {UNIFIED_INPUT_FILE}")
df = pd.read_csv(UNIFIED_INPUT_FILE, low_memory=False)
print(f"  Total cells: {len(df):,}")
print(f"  Columns: {len(df.columns)}")

# Verify required columns
required_cols = ['batch_id', 'x_centroid', 'y_centroid', 'phenotype_ml_mr']
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")
print(f"  Required columns present: {required_cols}")

# Check optional columns
optional_cols = ['patient', 'arm', 'response', 'timepoint', 'corrected', 'tissue_roi']
available_optional = [col for col in optional_cols if col in df.columns]
print(f"  Available optional columns: {available_optional}")

# Filter to corrected cells if available
if 'corrected' in df.columns:
    df = df[df['corrected'] == 1].copy()
    print(f"  Filtered to corrected cells: {len(df):,}")

# Display phenotype statistics
print(f"\nPhenotype Statistics (phenotype_ml_mr):")
phenotype_counts = df['phenotype_ml_mr'].value_counts()
for pheno, count in phenotype_counts.items():
    pct = count / len(df) * 100
    print(f"  {pheno}: {count:,} cells ({pct:.1f}%)")

print(f"\nAvailable markers (channels) - check your TIFF metadata")
print(f"  These will be extracted and available for visualization")

print(f"\nTotal unique batch_ids (slides): {df['batch_id'].nunique()}")
print(f"  Data loading complete!")

## Data Loading and Preprocessing

Load the unified CSV with cell phenotypes, coordinates, and batch IDs. Prepare for cell sampling.

## Cell Extraction Parameters

Adjust these parameters based on the data loaded above. After changing these, run the remaining cells without needing to reload data.

In [ ]:
# =============================================================================
# CELL EXTRACTION PARAMETERS - Adjust as needed
# =============================================================================

# Phenotype selection
PHENOTYPE_COLUMN = 'phenotype_ml_consensus'  # Column to filter phenotype
PHENOTYPE_OF_INTEREST = 'Mac'  # Choose from phenotypes listed above
# Set to None to sample from all phenotypes

# Sampling parameters
N_CELLS_TO_SAMPLE = 20  # Number of random cells to extract (e.g., 100 for comprehensive validation)
RANDOM_SEED = 42  # Seed for reproducibility

# Marker selection
# Set to None to use ALL available markers, or specify list of markers
MARKERS_OF_INTEREST = ['CD45', 'CD3e', 'CD4', 'CD8', 'FoxP3', 'CD68', 'CD11b', 'HLA-DR', 'CD11c', 'CD163']
# Common markers by phenotype:
#   Helper T: ['CD45', 'CD3e', 'CD4']
#   Cytotoxic T: ['CD45', 'CD3e', 'CD8a']
#   Macrophage: ['CD45', 'CD68', 'CD163']
#   Granulocyte: ['CD45', 'CD15', 'CD33']
#   B cell: ['CD45', 'CD20', 'CD19']
#   Treg: ['CD45', 'CD3e', 'CD4', 'FoxP3']
#   DC: ['CD45', 'CD11c', 'HLA-DR']

# Extract options
CELL_PADDING_PIXELS = 10  # Padding around cell polygon for context
MIN_MARKER_INTENSITY_THRESHOLD = 100  # Minimum intensity to consider marker present

# Figure options
FIGURE_DPI = 300
FIGURE_FORMAT = 'png'
FIGURE_FIGSIZE = (15, 10)  # (width, height) for cell display grid
CELLS_PER_ROW = 5  # How many cells to display per row
SHOW_OVERLAY = True  # Display overlay of all markers on same image

# =============================================================================
# NEW OPTION 1: Z-SCORE NORMALIZATION
# =============================================================================
# Choose between 'raw' or 'z-score' for marker intensity visualization
# - 'raw': Display pixel intensities as-is from the image
# - 'z-score': Normalize each marker per pixel using z-score within the batch_id
#   This highlights relative intensity differences within the cell and surrounding area
USE_Z_SCORE = False  # Set to True to use z-score normalized intensities

# =============================================================================
# NEW OPTION 2: VISUALIZE SURROUNDING CELLS
# =============================================================================
# Show surrounding cells and their phenotypes within a radius
SHOW_SURROUNDING_CELLS = True  # Set to True to display neighboring cells
SURROUNDING_CELLS_RADIUS = 40  # Radius in pixels to search for neighboring cells
# When enabled, displays:
#   - Center cell (PHENOTYPE_OF_INTEREST) with its polygon
#   - All surrounding cells within radius, colored by phenotype
#   - Marker channels with unique colors as overlay

# Marker colors for channel visualization
OVERLAY_COLORS = {
    'DAPI': '#0000FF',      # Blue
    'CD45': '#FF0000',      # Red
    'CD3e': '#00FF00',      # Green (Lime)
    'CD4': '#FFFF00',       # Yellow
    'CD8': '#00FFFF',       # Cyan
    'CD8a': '#00FFFF',      # Cyan (same as CD8)
    'FoxP3': '#FF00FF',     # Magenta
    'CD68': '#FF8C00',      # Dark Orange
    'CD163': '#FF69B4',     # Hot Pink
    'CD11b': '#8B4513',     # Saddle Brown
    'HLA-DR': '#9370DB',    # Medium Purple
    'CD15': '#DC143C',      # Crimson
    'CD31': '#FFD700',      # Gold
    'CD20': '#32CD32',      # Lime Green
    'HLA-A': '#3CB371',      # Medium Sea Green
    'CD11c': '#FF6347'      # Tomato
}

# Phenotype colors for cell boundary visualization (when showing surrounding cells)
PHENOTYPE_COLORS = {
    'Cytotoxic T': '#FF0000',      # Red
    'Helper T': '#0000FF',         # Blue
    'Treg': '#FF00FF',             # Magenta
    'Mac': '#FFA500',       # Orange
    'Granulocyte': '#00FFFF',      # Cyan
    'B cell': '#00FF00',           # Green
    'DC': '#FFFF00',               # Yellow
    'unclassified': '#808080'             # Gray
}

# Verbose output
VERBOSE = True

print("Cell Extraction Parameters Set:")
print(f"  Phenotype column: {PHENOTYPE_COLUMN}")
print(f"  Phenotype of interest: {PHENOTYPE_OF_INTEREST}")
print(f"  Number of cells to sample: {N_CELLS_TO_SAMPLE}")
print(f"  Markers to display: {MARKERS_OF_INTEREST}")
print(f"\n  Visualization options:")
print(f"    Z-score normalization: {USE_Z_SCORE}")
if USE_Z_SCORE:
    print(f"      (Markers will be z-normalized per pixel within batch_id)")
print(f"    Show surrounding cells: {SHOW_SURROUNDING_CELLS}")
if SHOW_SURROUNDING_CELLS:
    print(f"      Radius: {SURROUNDING_CELLS_RADIUS} pixels")
    print(f"      Shows neighboring cells with phenotype boundaries")
print(f"\n✓ Ready to run cell extraction!")
print("\n" + "="*80)
print("VISUALIZATION OPTIONS SUMMARY")
print("="*80)
print(f"\n✓ Z-SCORE NORMALIZATION: {USE_Z_SCORE}")
print(f"   - False (raw): Displays pixel intensities as-is from image")
print(f"   - True (z-score): Normalizes each marker within batch_id")
print(f"     → Highlights relative intensity differences between markers")
print(f"     → Better for comparing marker expression patterns")
print(f"\n✓ SURROUNDING CELLS: {SHOW_SURROUNDING_CELLS}")
print(f"   - False: Shows only the cell of interest")
print(f"   - True: Shows cell of interest + neighbors within {SURROUNDING_CELLS_RADIUS}px")
print(f"     → Displays phenotype labels for each cell")
print(f"     → Shows cell boundaries from GeoJSON")
print(f"     → Unique color per phenotype")
print(f"\n✓ MARKER COLORS: {len(OVERLAY_COLORS)} unique colors assigned")
print(f"   Markers: {', '.join(list(OVERLAY_COLORS.keys())[:5])}...")
print(f"\n✓ Ready to visualize!")

## Utility Functions for Cell Extraction

Define helper functions to extract cells, load images, and create visualizations.

In [ ]:
# =============================================================================
# UTILITY FUNCTIONS FOR CELL EXTRACTION
# =============================================================================

def load_geojson(filepath):
    """Load GeoJSON file and extract cell polygons."""
    try:
        with open(filepath, 'r') as f:
            geojson_data = json.load(f)
        return geojson_data
    except Exception as e:
        print(f"  Error loading GeoJSON {filepath}: {e}")
        return None

def find_cell_in_geojson(geojson_data, x_centroid, y_centroid, tolerance=20):
    """Find cell polygon in GeoJSON matching centroid coordinates.
    
    Args:
        geojson_data: Loaded GeoJSON dictionary
        x_centroid: X coordinate of cell centroid
        y_centroid: Y coordinate of cell centroid
        tolerance: Tolerance in pixels for coordinate matching
    
    Returns:
        Shapely Polygon if found, None otherwise
    """
    if geojson_data is None or 'features' not in geojson_data:
        return None
    
    point = Point(x_centroid, y_centroid)
    
    # First pass: exact point-in-polygon
    for feature in geojson_data['features']:
        coords = feature['geometry']['coordinates'][0]  # Exterior ring
        polygon = Polygon(coords)
        if polygon.contains(point):
            return polygon
    
    # Second pass: find nearest centroid within tolerance
    min_distance = float('inf')
    closest_polygon = None
    
    for feature in geojson_data['features']:
        if 'properties' in feature and 'centroid' in feature['properties']:
            centroid = feature['properties']['centroid']
            dist = np.sqrt((centroid[0] - x_centroid)**2 + (centroid[1] - y_centroid)**2)
            if dist < tolerance and dist < min_distance:
                min_distance = dist
                coords = feature['geometry']['coordinates'][0]
                closest_polygon = Polygon(coords)
    
    return closest_polygon

def extract_cell_image(roi_image, polygon, padding=10):
    """Extract cell image pixels within polygon from full ROI image.
    
    Args:
        roi_image: Full ROI image array (channels × height × width)
        polygon: Shapely Polygon defining cell boundary
        padding: Pixels to pad around polygon bounding box
    
    Returns:
        Extracted image patch (channels × patch_height × patch_width)
        Polygon mask for the cell
        Bounding box (x_min, y_min, x_max, y_max)
    """
    if polygon is None:
        return None, None, None
    
    # Get bounding box
    x_min, y_min, x_max, y_max = polygon.bounds
    x_min = max(0, int(x_min) - padding)
    y_min = max(0, int(y_min) - padding)
    x_max = min(roi_image.shape[2], int(x_max) + padding)
    y_max = min(roi_image.shape[1], int(y_max) + padding)
    
    # Extract patch
    patch = roi_image[:, y_min:y_max, x_min:x_max]
    
    # Create mask
    mask = np.zeros((y_max - y_min, x_max - x_min), dtype=bool)
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            if Point(x_min + j, y_min + i).within(polygon):
                mask[i, j] = True
    
    bbox = (x_min, y_min, x_max, y_max)
    return patch, mask, bbox

def load_roi_image(batch_id, roi_dir=ROI_IMAGES_DIR):
    """Load ROI TIFF image for given batch_id.
    
    Args:
        batch_id: Batch identifier (e.g., 'P11_roi_1')
        roi_dir: Directory containing ROI TIFF files
    
    Returns:
        Image array (channels × height × width) and channel names, or (None, None) if not found
    """
    # Search for matching ROI file
    roi_files = list(Path(roi_dir).glob(f"*{batch_id}*.ome.tiff"))
    
    if not roi_files:
        # Try alternative search patterns
        roi_files = list(Path(roi_dir).glob(f"*{batch_id}*.tiff"))
    
    if not roi_files:
        return None, None
    
    roi_path = roi_files[0]
    
    try:
        with tifffile.TiffFile(roi_path) as tif:
            # Load all channels
            image = np.array([page.asarray() for page in tif.pages])
            
            # Try to extract channel names from OME metadata
            channel_names = []
            if tif.ome_metadata:
                import xml.etree.ElementTree as ET
                try:
                    root = ET.fromstring(tif.ome_metadata)
                    namespace = {'ome': 'http://www.openmicroscopy.org/Schemas/OME/2016-06'}
                    channels = root.findall('.//ome:Channel', namespace)
                    channel_names = [ch.attrib.get('Name', f'Ch{i}') for i, ch in enumerate(channels)]
                except:
                    channel_names = [f'Ch{i}' for i in range(image.shape[0])]
            else:
                channel_names = [f'Ch{i}' for i in range(image.shape[0])]
            
            return image, channel_names
    except Exception as e:
        print(f"  Error loading ROI image {roi_path}: {e}")
        return None, None

def normalize_image(image, percentile_min=2, percentile_max=98):
    """Normalize image to 0-1 range using percentile clipping."""
    vmin = np.percentile(image, percentile_min)
    vmax = np.percentile(image, percentile_max)
    normalized = np.clip((image - vmin) / (vmax - vmin + 1e-6), 0, 1)
    return normalized

def hex_to_rgb(hex_color):
    """Convert hex color to normalized RGB (0-1 range)."""
    hex_color = hex_color.lstrip('#')
    if len(hex_color) == 6:
        return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    return (1.0, 1.0, 1.0)

def calculate_z_score(image_channel, mask=None):
    """Calculate z-score normalized image within mask or entire image."""
    if mask is not None:
        values = image_channel[mask]
    else:
        values = image_channel.flatten()
    
    mean = np.mean(values)
    std = np.std(values)
    
    if std < 1e-6:
        return np.zeros_like(image_channel, dtype=float)
    
    z_scored = (image_channel.astype(float) - mean) / std
    return z_scored

def find_surrounding_cells(geojson_data, center_x, center_y, radius):
    """Find all cells within radius of center point."""
    if geojson_data is None or 'features' not in geojson_data:
        return []
    
    surrounding = []
    radius_sq = radius ** 2
    
    for feature in geojson_data['features']:
        coords = feature.get('geometry', {}).get('coordinates', [[]])
        if not coords or not coords[0] or len(coords[0]) < 3:
            continue
        
        try:
            polygon = Polygon(coords[0])
            centroid = polygon.centroid
            dist = np.sqrt((centroid.x - center_x)**2 + (centroid.y - center_y)**2)
            
            if dist <= radius:
                surrounding.append({
                    'polygon': polygon,
                    'centroid': (centroid.x, centroid.y),
                    'distance': dist
                })
        except:
            continue
    
    return surrounding


def create_phenotype_legend(output_dir=OUTPUT_DIR, figure_dpi=FIGURE_DPI):
    """Create a standalone legend PNG for phenotype colors."""
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(PHENOTYPE_COLORS))
    ax.axis('off')
    
    # Title
    fig.suptitle('Phenotype Color Legend', fontsize=16, fontweight='bold', y=0.98)
    
    # Create legend entries
    y_pos = len(PHENOTYPE_COLORS) - 0.5
    for phenotype, color_hex in PHENOTYPE_COLORS.items():
        color_rgb = hex_to_rgb(color_hex)
        # Draw color box
        rect = plt.Rectangle((0.5, y_pos - 0.35), 0.5, 0.7, 
                            facecolor=color_rgb, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        # Add phenotype label
        ax.text(1.5, y_pos, phenotype, fontsize=12, va='center', fontweight='bold')
        y_pos -= 1
    
    plt.tight_layout()
    legend_path = Path(output_dir) / 'phenotype_color_legend.png'
    plt.savefig(legend_path, dpi=figure_dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    return legend_path


def create_phenotype_legend(output_dir=OUTPUT_DIR, figure_dpi=FIGURE_DPI):
    """Create a standalone legend PNG for phenotype colors."""
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(PHENOTYPE_COLORS))
    ax.axis('off')
    
    fig.suptitle('Phenotype Color Legend', fontsize=16, fontweight='bold', y=0.98)
    
    y_pos = len(PHENOTYPE_COLORS) - 0.5
    for phenotype, color_hex in PHENOTYPE_COLORS.items():
        color_rgb = hex_to_rgb(color_hex)
        rect = plt.Rectangle((0.5, y_pos - 0.35), 0.5, 0.7, 
                            facecolor=color_rgb, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(1.5, y_pos, phenotype, fontsize=12, va='center', fontweight='bold')
        y_pos -= 1
    
    plt.tight_layout()
    legend_path = Path(output_dir) / 'phenotype_color_legend.png'
    plt.savefig(legend_path, dpi=figure_dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    return legend_path


## Sample Cells and Extract Markers

Randomly sample cells from selected phenotype and extract marker images.

In [ ]:
# =============================================================================
# STEP 2: SAMPLE CELLS
# =============================================================================
print("\n" + "="*80)
print("STEP 2: SAMPLING CELLS")
print("="*80)

# Set random seed
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Filter to phenotype of interest
if PHENOTYPE_OF_INTEREST:
    df_sample = df[df[PHENOTYPE_COLUMN] == PHENOTYPE_OF_INTEREST].copy()
    print(f"\nFiltered to phenotype: {PHENOTYPE_OF_INTEREST}")
    print(f"  Cells available: {len(df_sample):,}")
else:
    df_sample = df.copy()
    print(f"\nNo phenotype filter (using all cells)")
    print(f"  Cells available: {len(df_sample):,}")

# Check we have enough cells
if len(df_sample) < N_CELLS_TO_SAMPLE:
    print(f"  WARNING: Only {len(df_sample)} cells available, requesting {N_CELLS_TO_SAMPLE}")
    N_CELLS_TO_SAMPLE = len(df_sample)
    print(f"  Using all {N_CELLS_TO_SAMPLE} available cells")

# Random sample
sampled_cells = df_sample.sample(n=N_CELLS_TO_SAMPLE, random_state=RANDOM_SEED).reset_index(drop=True)

# OPTIMIZATION: Sort by batch_id to process cells from same image together
sampled_cells = sampled_cells.sort_values('batch_id').reset_index(drop=True)

print(f"\nSampled cells: {len(sampled_cells)}")
print(f"  From {sampled_cells['batch_id'].nunique()} different slides")
print(f"  ✓ Sorted by batch_id for efficient batch processing")

# Display sampling statistics
print(f"\nSampling by batch_id:")
batch_counts = sampled_cells['batch_id'].value_counts().head(10)
for batch_id, count in batch_counts.items():
    print(f"  {batch_id}: {count} cells")

if len(batch_counts) > 10:
    print(f"  ... and {len(sampled_cells['batch_id'].unique()) - 10} more slides")

## Extract Cell Images and Process Markers

For each sampled cell, load segmentation polygon, extract cell image, and process selected markers.

In [ ]:
# =============================================================================
# STEP 3: EXTRACT, VISUALIZE, AND SAVE CELLS (STREAMING)
# =============================================================================
print("\n" + "="*80)
print("STEP 3: EXTRACTING AND VISUALIZING CELLS")
print("="*80)

# Cache for loaded GeoJSON files and ROI images to avoid redundant I/O
geojson_cache = {}
roi_image_cache = {}

extracted_count = 0
failed_cells = []

print(f"\nProcessing {len(sampled_cells)} cells...\n")

for idx, row in tqdm(sampled_cells.iterrows(), total=len(sampled_cells), desc="Processing cells"):
    batch_id = row['batch_id']
    x_centroid = row['x_centroid']
    y_centroid = row['y_centroid']
    
    try:
        # Load GeoJSON if not cached
        if batch_id not in geojson_cache:
            geojson_files = list(Path(SEGMENTATION_DIR).glob(f"*{batch_id}*.geojson"))
            if not geojson_files:
                geojson_files = list(Path(SEGMENTATION_DIR).glob(f"*{batch_id.split('_')[0]}*.geojson"))
            if geojson_files:
                geojson_data = load_geojson(str(geojson_files[0]))
                geojson_cache[batch_id] = geojson_data
            else:
                geojson_cache[batch_id] = None
        
        geojson_data = geojson_cache[batch_id]
        
        # Find cell polygon
        polygon = find_cell_in_geojson(geojson_data, x_centroid, y_centroid)
        if polygon is None:
            failed_cells.append({'idx': idx, 'batch_id': batch_id, 'reason': 'polygon_not_found'})
            continue
        
        # Load ROI image if not cached
        if batch_id not in roi_image_cache:
            image, channel_names = load_roi_image(batch_id)
            roi_image_cache[batch_id] = (image, channel_names)
        
        image, channel_names = roi_image_cache[batch_id]
        
        if image is None:
            failed_cells.append({'idx': idx, 'batch_id': batch_id, 'reason': 'image_not_found'})
            continue
        
        # Extract cell patch
        patch, mask, bbox = extract_cell_image(image, polygon, padding=CELL_PADDING_PIXELS)
        if patch is None:
            failed_cells.append({'idx': idx, 'batch_id': batch_id, 'reason': 'extraction_failed'})
            continue
        
        # Select markers
        if MARKERS_OF_INTEREST is None:
            selected_markers = channel_names
        else:
            selected_markers = [m for m in MARKERS_OF_INTEREST if m in channel_names]
        
        # Find surrounding cells if requested
        surrounding_cells = []
        if SHOW_SURROUNDING_CELLS:
            surrounding_cells = find_surrounding_cells(geojson_data, x_centroid, y_centroid, SURROUNDING_CELLS_RADIUS)
            for sc in surrounding_cells:
                sc_x, sc_y = sc['centroid']
                matching_cells = df[(df['batch_id'] == batch_id) & 
                                   (np.abs(df['x_centroid'] - sc_x) < 5) & 
                                   (np.abs(df['y_centroid'] - sc_y) < 5)]
                if len(matching_cells) > 0:
                    sc['phenotype'] = matching_cells.iloc[0].get(PHENOTYPE_COLUMN, 'Unknown')
                else:
                    sc['phenotype'] = 'Unknown'
        
        # ===== VISUALIZE THIS CELL =====
        roi_image = image
        channel_z_scores = {}
        
        if USE_Z_SCORE:
            for marker in selected_markers:
                if marker in channel_names:
                    ch_idx = channel_names.index(marker)
                    ch_full = roi_image[ch_idx].astype(float)
                    z_scored = calculate_z_score(ch_full)
                    z_scored = np.clip((z_scored - z_scored.min()) / (z_scored.max() - z_scored.min() + 1e-6), 0, 1)
                    channel_z_scores[marker] = z_scored
        
        n_markers = len(selected_markers)
        n_marker_cols = n_markers + (1 if SHOW_OVERLAY else 0)
        n_surrounding = len(surrounding_cells) if SHOW_SURROUNDING_CELLS else 0
        
        if SHOW_SURROUNDING_CELLS and n_surrounding > 0:
            n_rows = 2
            n_cols = n_marker_cols
        else:
            n_rows = 1
            n_cols = n_marker_cols
        
        fig = plt.figure(figsize=(n_cols * 3.5, n_rows * 3.5))
        ax_idx = 1
        
        # Row 1: Individual markers for center cell
        for marker in selected_markers:
            if marker in channel_names:
                ax = fig.add_subplot(n_rows, n_cols, ax_idx)
                ax_idx += 1
                ch_idx = channel_names.index(marker)
                bbox_x_min, bbox_y_min, bbox_x_max, bbox_y_max = bbox
                if USE_Z_SCORE:
                    ch_normalized = channel_z_scores[marker][bbox_y_min:bbox_y_max, bbox_x_min:bbox_x_max]
                else:
                    ch_image = patch[ch_idx].astype(float)
                    ch_normalized = normalize_image(ch_image)
                masked_image = np.zeros((*ch_normalized.shape, 3), dtype=float)
                if marker in OVERLAY_COLORS:
                    color = hex_to_rgb(OVERLAY_COLORS[marker])
                else:
                    color = (1.0, 1.0, 1.0)
                for c in range(3):
                    masked_image[mask, c] = ch_normalized[mask] * color[c]
                ax.imshow(masked_image)
                ax.set_title(f'{marker}', fontweight='bold', fontsize=10)
                ax.axis('off')
        
        # Overlay for center cell
        if SHOW_OVERLAY and n_markers > 0:
            ax = fig.add_subplot(n_rows, n_cols, ax_idx)
            ax_idx += 1
            overlay = np.zeros((*mask.shape, 3), dtype=float)
            for marker in selected_markers:
                if marker in channel_names:
                    ch_idx = channel_names.index(marker)
                    bbox_x_min, bbox_y_min, bbox_x_max, bbox_y_max = bbox
                    if USE_Z_SCORE:
                        ch_normalized = channel_z_scores[marker][bbox_y_min:bbox_y_max, bbox_x_min:bbox_x_max]
                    else:
                        ch_image = patch[ch_idx].astype(float)
                        ch_normalized = normalize_image(ch_image)
                    if marker in OVERLAY_COLORS:
                        color = hex_to_rgb(OVERLAY_COLORS[marker])
                    else:
                        color = (1.0, 1.0, 1.0)
                    for c in range(3):
                        overlay[mask, c] += ch_normalized[mask] * color[c]
            overlay_max = overlay.max()
            if overlay_max > 0:
                overlay = overlay / overlay_max
            ax.imshow(overlay)
            ax.set_title('Overlay (all markers)', fontweight='bold', fontsize=10)
            ax.axis('off')
        
        # Row 2: Surrounding cells
        if SHOW_SURROUNDING_CELLS and len(surrounding_cells) > 0:
            all_polys = [polygon]
            for sc in surrounding_cells:
                all_polys.append(sc['polygon'])
            combined_bounds = unary_union(all_polys).bounds
            x_min = max(0, int(combined_bounds[0]) - 20)
            y_min = max(0, int(combined_bounds[1]) - 20)
            x_max = min(roi_image.shape[2], int(combined_bounds[2]) + 20)
            y_max = min(roi_image.shape[1], int(combined_bounds[3]) + 20)
            
            for marker_idx, marker in enumerate(selected_markers[:n_marker_cols - 1]):
                if marker_idx >= n_cols:
                    break
                if marker in channel_names:
                    ax = fig.add_subplot(n_rows, n_cols, n_cols + marker_idx + 1)
                    if USE_Z_SCORE:
                        ch_normalized = channel_z_scores[marker][y_min:y_max, x_min:x_max]
                    else:
                        ch_idx = channel_names.index(marker)
                        ch_image_region = roi_image[ch_idx, y_min:y_max, x_min:x_max].astype(float)
                        ch_normalized = normalize_image(ch_image_region)
                    if marker in OVERLAY_COLORS:
                        color = hex_to_rgb(OVERLAY_COLORS[marker])
                    else:
                        color = (1.0, 1.0, 1.0)
                    colored_image = np.zeros((*ch_normalized.shape, 3), dtype=float)
                    for c in range(3):
                        colored_image[:, :, c] = ch_normalized * color[c]
                    ax.imshow(colored_image)
                    ax.set_title(f'{marker} (surrounding)', fontweight='bold', fontsize=9)
                    ax.axis('off')
                    for poly_idx, polygon_item in enumerate(all_polys):
                        x_poly, y_poly = polygon_item.exterior.xy
                        x_poly = np.array(x_poly) - x_min
                        y_poly = np.array(y_poly) - y_min
                        if poly_idx == 0:
                            ax.plot(x_poly, y_poly, 'c-', linewidth=2.5, alpha=0.9)
                        else:
                            sc_phenotype = surrounding_cells[poly_idx - 1].get('phenotype', 'Unknown')
                            poly_color = PHENOTYPE_COLORS.get(sc_phenotype, '#FFFFFF')
                            ax.plot(x_poly, y_poly, color=poly_color, linewidth=1.5, alpha=0.8)
            
            if SHOW_OVERLAY and len(selected_markers) > 0:
                ax = fig.add_subplot(n_rows, n_cols, n_cols + n_marker_cols)
                overlay_surr = np.zeros((y_max - y_min, x_max - x_min, 3), dtype=float)
                for marker in selected_markers:
                    if marker in channel_names:
                        if USE_Z_SCORE:
                            ch_normalized = channel_z_scores[marker][y_min:y_max, x_min:x_max]
                        else:
                            ch_idx = channel_names.index(marker)
                            ch_image_region = roi_image[ch_idx, y_min:y_max, x_min:x_max].astype(float)
                            ch_normalized = normalize_image(ch_image_region)
                        if marker in OVERLAY_COLORS:
                            color = hex_to_rgb(OVERLAY_COLORS[marker])
                        else:
                            color = (1.0, 1.0, 1.0)
                        for c in range(3):
                            overlay_surr[:, :, c] += ch_normalized * color[c]
                overlay_max = overlay_surr.max()
                if overlay_max > 0:
                    overlay_surr = overlay_surr / overlay_max
                ax.imshow(overlay_surr)
                ax.set_title('Overlay (surrounding)', fontweight='bold', fontsize=9)
                ax.axis('off')
                for poly_idx, polygon_item in enumerate(all_polys):
                    x_poly, y_poly = polygon_item.exterior.xy
                    x_poly = np.array(x_poly) - x_min
                    y_poly = np.array(y_poly) - y_min
                    if poly_idx == 0:
                        ax.plot(x_poly, y_poly, 'c-', linewidth=2.5, label='Cell of interest')
                    else:
                        sc_phenotype = surrounding_cells[poly_idx - 1].get('phenotype', 'Unknown')
                        poly_color = PHENOTYPE_COLORS.get(sc_phenotype, '#FFFFFF')
                        ax.plot(x_poly, y_poly, color=poly_color, linewidth=1.5, alpha=0.8, label=sc_phenotype)
                # Legend removed - see phenotype_color_legend.png for reference
        
        normalization_label = 'Z-Score' if USE_Z_SCORE else 'Raw'
        if SHOW_SURROUNDING_CELLS:
            row_desc = f'Row1: Center cell | Row2: +{n_surrounding} cells'
        else:
            row_desc = 'Center cell only'
        fig.suptitle(f"{row.get(PHENOTYPE_COLUMN, 'Unknown')} | ({x_centroid:.0f}, {y_centroid:.0f}) | {batch_id} | {normalization_label} | {row_desc}", fontsize=12, fontweight='bold', y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        
        # Save figure
        fig_filename = f"cell_{idx:04d}_{batch_id}_markers.{FIGURE_FORMAT}"
        fig_path = OUTPUT_DIR / fig_filename
        plt.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        
        extracted_count += 1
        if VERBOSE:
            print(f"  ✓ Saved: {fig_filename}")
        
    except Exception as e:
        failed_cells.append({'idx': idx, 'batch_id': batch_id, 'reason': f'exception: {str(e)}'})
        plt.close('all')

# Generate phenotype color legend
legend_path = create_phenotype_legend()
print(f"\n✓ Created phenotype color legend: {legend_path}")

# Generate phenotype color legend
legend_path = create_phenotype_legend()
print(f"\n✓ Created phenotype color legend: {legend_path}")

print(f"\n\nExtraction & Visualization Summary:")
print(f"  Successfully processed: {extracted_count} cells")
print(f"  Failed: {len(failed_cells)} cells")

if failed_cells:
    print(f"\n  Failed cell reasons:")
    for reason in set([f['reason'] for f in failed_cells]):
        count = len([f for f in failed_cells if f['reason'] == reason])
        print(f"    {reason}: {count}")

print(f"\n{'='*80}")
print(f"STEP 3 COMPLETE!")
print(f"Generated {len([f for f in OUTPUT_DIR.glob(f'*.{FIGURE_FORMAT}')])} figures")
print(f"Output directory: {OUTPUT_DIR}")
print(f"{'='*80}")

## Visualize Extracted Cells with Markers

Create figures displaying extracted cell images with individual marker channels and overlays.

## Generate Summary Report

Create a comprehensive summary of cell extraction and marker analysis.

In [ ]:
# =============================================================================
# STEP 5: SUMMARY REPORT
# =============================================================================
print("\n" + "="*80)
print("STEP 5: SUMMARY REPORT")
print("="*80)

report = f"""

{'='*80}
CELL MARKER VISUALIZATION SUMMARY
{'='*80}

Configuration:
  Phenotype column: {PHENOTYPE_COLUMN}
  Phenotype of interest: {PHENOTYPE_OF_INTEREST}
  Cells to sample: {N_CELLS_TO_SAMPLE}
  Random seed: {RANDOM_SEED}
  Markers of interest: {MARKERS_OF_INTEREST}
  Cell padding: {CELL_PADDING_PIXELS} pixels

Extraction Results:
  Total cells sampled: {len(sampled_cells)}
  Successfully extracted: {len(extracted_cells)}
  Failed: {len(failed_cells)}
  Success rate: {len(extracted_cells) / len(sampled_cells) * 100:.1f}%

Cell Distribution:
  Unique slides: {len(set([c['batch_id'] for c in extracted_cells]))}
  Average markers per cell: {np.mean([len(c['selected_markers']) for c in extracted_cells]):.1f}

Output:
  Output directory: {OUTPUT_DIR}
  Figure format: {FIGURE_FORMAT}
  Figure DPI: {FIGURE_DPI}

{'='*80}
Analysis Complete!
{'='*80}
"""

print(report)

# Save report to file
report_path = OUTPUT_DIR / 'extraction_summary_report.txt'
with open(report_path, 'w') as f:
    f.write(report)
print(f"\nReport saved: {report_path}")

## Export Cell Data for Further Analysis

Create a CSV file with metadata for all extracted cells for downstream analysis or annotation.

In [ ]:
# =============================================================================
# STEP 6: EXPORT CELL METADATA
# =============================================================================
print("\n" + "="*80)
print("STEP 6: EXPORTING CELL METADATA")
print("="*80)

if len(extracted_cells) > 0:
    # Create metadata dataframe
    cell_metadata = []
    
    for cell in extracted_cells:
        metadata_row = {
            'cell_id': cell['cell_id'],
            'batch_id': cell['batch_id'],
            'x_centroid': cell['x_centroid'],
            'y_centroid': cell['y_centroid'],
            'phenotype': cell['phenotype'],
            'bbox_x_min': cell['bbox'][0],
            'bbox_y_min': cell['bbox'][1],
            'bbox_x_max': cell['bbox'][2],
            'bbox_y_max': cell['bbox'][3],
            'patch_height': cell['patch'].shape[1],
            'patch_width': cell['patch'].shape[2],
            'n_channels': cell['patch'].shape[0],
            'channels': ','.join(cell['channel_names']),
            'selected_markers': ','.join(cell['selected_markers']),
            'n_markers_displayed': len(cell['selected_markers'])
        }
        cell_metadata.append(metadata_row)
    
    df_cells = pd.DataFrame(cell_metadata)
    
    # Save to CSV
    csv_path = OUTPUT_DIR / 'extracted_cells_metadata.csv'
    df_cells.to_csv(csv_path, index=False)
    print(f"\nExported cell metadata CSV: {csv_path}")
    print(f"  Rows: {len(df_cells)}")
    print(f"  Columns: {len(df_cells.columns)}")
    
    print(f"\nMetadata preview:")
    print(df_cells.head(10).to_string())
else:
    print("\nNo cells extracted, skipping metadata export.")